# Hierarchical Multi-Agent RAG (Orchestrator-Worker Pattern)

## Table of Contents
1. [What Is Hierarchical Multi-Agent RAG?](#what-is-hierarchical-multi-agent-rag)
2. [How It Works: The Complete Process](#how-it-works-the-complete-process)
3. [Why and When to Use It](#why-and-when-to-use-it)
4. [Full Workflow Diagram](#full-workflow-diagram)
5. [Pros and Cons](#pros-and-cons)
6. [Building from Scratch in LangGraph](#building-from-scratch-in-langgraph)
7. [Enterprise M&A Case Study Overview](#enterprise-ma-case-study-overview)

---

## What Is Hierarchical Multi-Agent RAG?

In standard single-agent or routing RAG architectures, a single Language Model is responsible for parsing a query, selecting the correct tool or search index, retrieving data chunks, reading through all the retrieved context, and synthesizing the final answer. 

While this works for simple lookups, it suffers from severe limitations when applied to **complex, multi-domain, or enterprise-scale problems**. For example, if a query requires reading across SEC filings, technical patent databases, and legal case law, a single LLM's context window can be easily overwhelmed by noise. Worse, the LLM may fail to coordinate the distinct search strategies required for each specific source.

**Hierarchical Multi-Agent RAG** addresses this by applying a corporate organizational structure to information retrieval. It establishes a two-tiered hierarchy:
1. **Orchestrator (The Supervisor / Planner)**: A high-level agent responsible for query decomposition, dynamic routing, worker task assignment, quality assurance, and final synthesis.
2. **Workers (Specialists)**: A set of highly focused, localized agents (often modeled as subgraphs or ReAct agents) that are domain-experts in a single database or knowledge repository. Each worker possesses its own specialized retrieval tools, embedding models, and search logic.

```
                  ┌───────────────────────────────┐
                  │       👤 User Query           │
                  └───────────────┬───────────────┘
                                  │
                                  ▼
                  ┌───────────────────────────────┐
                  │   🧠 M&A Deal Supervisor      │ (Orchestrator / Router)
                  └──────┬────────┬────────┬──────┘
                         │        │        │
         ┌───────────────┘        │        └───────────────┐
         ▼                        ▼                        ▼
┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐
│ 📊 Financial RAG│      │ 🔬 Patent RAG   │      │ ⚖️ Legal RAG     │ (Specialist Workers)
│ (SEC Filings DB)│      │ (Tech Papers DB)│      │ (Compliance DB) │
└─────────────────┘      └─────────────────┘      └─────────────────┘
```

---

## How It Works: The Complete Process

The execution of a Hierarchical Multi-Agent RAG flow proceeds in five distinct phases:

### 1. Query Decomposition & Planning
When the Orchestrator receives a complex multi-part query, it does not immediately execute a search. Instead, it evaluates the query against the capabilities of its available workers. It breaks the user query down into distinct, specialized sub-queries.
* *Example User Query*: *"Evaluate TechCorp's 2025 financial health, check their recent optical patents, and verify if they have any pending FTC antitrust litigation."*
* *Decomposed Sub-Queries*:
  1. Financial: Retrieve 2025 income statements and balance sheets for TechCorp.
  2. IP/Tech: Search patent records for TechCorp's optical engineering filings.
  3. Legal: Search federal compliance databases for FTC antitrust lawsuits against TechCorp.

### 2. Dynamic Routing & Worker Dispatch
The Orchestrator maintains state and uses **Structured Outputs** (via tools or Pydantic schemas) to determine which worker to invoke next. In a parallel setting, it dispatches all relevant workers concurrently. In a sequential setting (e.g., if finding the lawsuits depends on discovering the patent names first), it routes step-by-step.

### 3. Isolated Specialized Retrieval (Worker Execution)
Each worker executes in its own isolated node or subgraph:
* It reformulates the incoming sub-query specifically for its targeted vector database.
* It invokes specialized retrieval tools (e.g., dense vector search, hybrid keyword search, or BM25).
* It scores, grades, and filters the retrieved chunks, removing low-relevance noise.
* It synthesizes a concise, fact-grounded **Domain Briefing** containing only high-value facts and citations.

### 4. Information Hiding & Communication
To prevent context window bloat and "lost-in-the-middle" issues for the Orchestrator, Hierarchical RAG utilizes an **Information Hiding** pattern. Instead of sending raw document chunks (thousands of tokens of text) back to the Orchestrator, the workers compile a concise summary report. The Orchestrator *only* sees the final summary briefings, keeping its context clean and focused on high-level planning.

### 5. Multi-Source Synthesis & Citation Tracing
Once all workers have reported back, the Orchestrator aggregates the domain briefings. It performs a final reasoning step to combine the heterogeneous information, traces citations back to their source agents, and writes a comprehensive, multi-perspective response to the user.

---

## Why and When to Use It

### When to Use Hierarchical RAG:
* **Heterogeneous Data Formats**: When your enterprise data is scattered across completely different structures (e.g., highly structured SEC financial tables, unstructured scientific patent publications, and semi-structured legal PDF briefs).
* **Context Overload**: When retrieving raw documents from all domains simultaneously would exceed LLM context limits or dilute attention, leading to hallucinations.
* **Isolated Search Strategies**: When different knowledge bases require unique search mechanisms (e.g., financial data needs precise SQL or table lookup; technology papers need hybrid semantic search; legal data needs Boolean keyword queries).
* **High-Stakes Decision Support**: When errors or missed documents carry significant financial or regulatory risks (e.g., M&A due diligence, legal audits, medical diagnostic reviews).

> [!TIP]
> If your system is retrieving from a single, unified database of general corporate wiki pages, standard RAG is sufficient. Transition to Hierarchical RAG only when you have **distinct, domain-specific databases** that require specialized retrieval logic.

---

## Full Workflow Diagram

The following diagram illustrates the complete control flow, execution loop, and data separation between the Orchestrator and the specialized Retrieval Workers.

```mermaid
sequenceDiagram
    autonumber
    actor User as Corporate User
    participant Supervisor as 🧠 M&A Deal Supervisor<br/>(Orchestrator Node)
    participant FinAgent as 📊 Financial Specialist<br/>(Worker Agent)
    participant IPAgent as 🔬 Intellectual Property Specialist<br/>(Worker Agent)
    participant LegalAgent as ⚖️ Legal & Compliance Specialist<br/>(Worker Agent)
    
    User->>Supervisor: "Analyze TechCorp's due diligence metrics (Financial, Patents, Lawsuits)"
    Note over Supervisor: Decomposes query into domain sub-queries.<br/>Determines routing sequence.
    
    rect rgb(240, 248, 255)
        Note over Supervisor, FinAgent: Step 1: Financial Assessment
        Supervisor->>FinAgent: Invoke sub-query: "Analyze TechCorp 2025 financials"
        Note over FinAgent: Vector Search on SEC Filings DB.<br/>Filters top chunks.<br/>Compiles Financial Briefing.
        FinAgent-->>Supervisor: Return concise Financial Briefing with citations
    end
    
    rect rgb(245, 240, 255)
        Note over Supervisor, IPAgent: Step 2: Intellectual Property Check
        Supervisor->>IPAgent: Invoke sub-query: "Search optical technology patents"
        Note over IPAgent: Vector Search on Tech Publications DB.<br/>Grades technical relevance.<br/>Compiles IP Briefing.
        IPAgent-->>Supervisor: Return concise IP Briefing with citations
    end

    rect rgb(255, 240, 240)
        Note over Supervisor, LegalAgent: Step 3: Legal & Regulatory Review
        Supervisor->>LegalAgent: Invoke sub-query: "Identify active lawsuits or FTC disputes"
        Note over LegalAgent: Search Legal Database.<br/>Extracts active litigation details.<br/>Compiles Legal Briefing.
        LegalAgent-->>Supervisor: Return concise Legal Briefing with citations
    end

    Note over Supervisor: Aggregates Financial, IP, and Legal Briefings.<br/>Validates cross-reference citations.
    Supervisor-->>User: Deliver comprehensive M&A Due Diligence Report (Fact-grounded)
```

---

## Pros and Cons

| Feature | Pros | Cons |
| :--- | :--- | :--- |
| **Context Optimization** | 🚀 **Excellent**. Information hiding ensures the central planner's context window stays clean and free of raw retrieval noise. | ⚠️ **Slightly higher total token usage** due to workers performing intermediary synthesis. |
| **Retrieval Precision** | 🎯 **High**. Each agent uses tools optimized for its database (e.g. dense vector search, hybrid retrieval, SQL). | 🛠️ **Higher upfront setup** needed to tune individual worker tools and prompts. |
| **System Scalability** | 📈 **Seamless**. You can add new workers (e.g. "HR Expert", "Supply Chain Expert") by modifying the supervisor's router schema. | 🔀 **Supervisor complexity** increases as routing options grow. |
| **Failure Isolation** | 🛡️ **Robust**. A failure in the legal database search does not break the financial analysis; the supervisor can gracefully report legal data missing. | 🔄 **Propagation Risks**. If the supervisor fails to decompose the initial query properly, workers will retrieve irrelevant data. |
| **Execution Latency** | ⚡ **Parallelizable**. Independent queries run simultaneously in sub-graphs. | 🐢 **Sequential overhead** if agent steps have rigid dependency chains. |

---

## Building from Scratch in LangGraph

To build a robust Hierarchical Multi-Agent RAG system in LangGraph, you must master three main components:

### 1. Define the Global & Local States
We maintain a parent state (`M&ADiligenceState`) to manage the high-level communications, and local worker states (or parameters) to handle retrieval specifics.

```python
from langgraph.graph import MessagesState

class M&ADiligenceState(MessagesState):
    # Tracks which agent the supervisor chooses to route to next
    next_agent: str
    # Holds compiled briefings returned by specialist workers
    financial_brief: str
    ip_brief: str
    legal_brief: str
```

### 2. Structured Orchestrator Routing
Instead of raw text output, the Orchestrator uses Pydantic structured output (`with_structured_output`) to decide the next step. This guarantees deterministic state transitions.

```python
from pydantic import BaseModel, Field
from typing import Literal

class RouterDecision(BaseModel):
    next: Literal["financial_worker", "ip_worker", "legal_worker", "FINISH"] = Field(
        description="Choose the next expert worker to consult, or FINISH if you have all facts."
    )
    sub_query: str = Field(description="The specific question/query formulated for the selected specialist worker.")
    reasoning: str = Field(description="Explanation of why this path was chosen.")
```

### 3. Information Hiding Wrapper
Rather than registering a worker's raw database search directly as a node in the supervisor graph, we wrap the worker agent. The wrapper runs the worker, extracts the structured briefing, and returns it to the parent state:

```python
def run_financial_worker(state: M&ADiligenceState) -> dict:
    # 1. Fetch current sub-query formulated by supervisor
    query = state["messages"][-1].content
    # 2. Invoke specialized agent (running tools, vector retrievers, re-ranking)
    result = financial_agent_compiled.invoke({"messages": [HumanMessage(content=query)]})
    # 3. Return summary report to CEO state (Information Hiding)
    final_brief = result["messages"][-1].content
    return {
        "messages": [AIMessage(content=f"[FINANCIAL REPORT]: {final_brief}", name="FinancialWorker")],
        "financial_brief": final_brief
    }
```

---

## Enterprise M&A Case Study Overview

In our code implementation (`24_hierarchical_rag.py`), we simulate a real-world mergers and acquisitions due diligence process for a fictitious target company, **"QuantumTech Inc."**

We build three mock vector databases containing rich, detailed documents:
1. **SEC Financial Database**: Storing annual balance sheets, operating revenues, debt ratios, and EBITDA tables for QuantumTech.
2. **Patent Database**: Storing patents on quantum encryption chips, silicon photonics, and hardware keys.
3. **Legal Litigation Database**: Storing ongoing class-action lawsuits, FTC antitrust reviews, and compliance records.

The **M&A Deal Supervisor** coordinates searches across these three distinct sources, digests their findings, and builds a professional, integrated investment risk report complete with domain citations.

## Complete Implementation Code

In [ ]:
"""
Hierarchical Multi-Agent RAG Pattern - LangGraph Implementation
==============================================================

This script implements a Hierarchical (Orchestrator-Worker) Multi-Agent RAG pattern:
1. M&A Deal Supervisor (Orchestrator): Receives a complex due diligence request,
   decomposes it, dynamically routes to specialized worker sub-agents, and
   synthesizes a high-level fact-grounded investment briefing.
2. Financial RAG Agent (Worker): Specialized in balance sheets, revenues, and debt metrics.
3. IP & Technology RAG Agent (Worker): Specialized in patents, whitepapers, and hardware claims.
4. Legal & Compliance RAG Agent (Worker): Specialized in litigation, FTC antitrust, and audits.

Key Concepts:
- Dynamic Routing with Structured Output: The Supervisor uses Pydantic to select the next worker and draft sub-queries.
- Information Hiding: Workers run full ReAct retrieval loops locally, but only return a synthesized, high-level summary briefing back to the Supervisor graph. This prevents context bloat.
- Isolated Specialized Search Tools: Each worker agent interacts with its own separate database mock.
"""


In [1]:

import os
from typing import Literal, List, Dict, Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import create_react_agent

ModuleNotFoundError: No module named 'langchain_core'

In [ ]:

# Load environment variables
load_dotenv()


In [ ]:


# Set up models with dual-provider support (OpenAI and Groq)
if os.environ.get("OPENAI_API_KEY"):
    print("[INFO] Detected OpenAI API Key. Running with OpenAI (gpt-4o & gpt-4o-mini)...")
    from langchain_openai import ChatOpenAI
    orchestrator_model = ChatOpenAI(model='gpt-4o', temperature=0)
    worker_model = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)
elif os.environ.get("GROQ_API_KEY"):
    print("[INFO] Detected Groq API Key. Running with Groq (llama-3.3-70b-versatile & llama-3.1-8b-instant)...")
    from langchain_groq import ChatGroq
    # Llama 3.3 70B is an exceptional, fast model for planning and structured output
    orchestrator_model = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
    # Llama 3.1 8B is perfect, fast, and lightweight for worker agents
    worker_model = ChatGroq(model='llama-3.1-8b-instant', temperature=0.2)
else:
    raise ValueError("Missing credentials. Please set either OPENAI_API_KEY or GROQ_API_KEY in your environment / .env file.")


In [ ]:

# ==============================================================================
# Step 1: Define Mock Knowledge Bases (Corporate Databases)
# ==============================================================================

# 1. Financial Database (SEC 10-K / Balance Sheet Filings)
FINANCIAL_DB = [
    {
        "company": "QuantumTech Inc.",
        "year": 2025,
        "document": (
            "QuantumTech Inc. SEC Form 10-K (FY2025):\n"
            "- Total Revenue: $142.5 Million USD (up 28% year-over-year from $111.3 Million in 2024).\n"
            "- Gross Margin: 64.2% driven by enterprise quantum SaaS subscriptions.\n"
            "- Operating Income (EBITDA): $38.4 Million USD.\n"
            "- Cash and Equivalents: $45.2 Million USD.\n"
            "- Total Debt Liabilities: $12.0 Million USD (Long-term senior convertible notes).\n"
            "- Principal Risks: High R&D overhead ($52.0M spent in 2025) and reliance on specialized chip fabrication foundries."
        ),
        "source": "SEC-10K-FY2025"
    },
    {
        "company": "QuantumTech Inc.",
        "year": 2024,
        "document": (
            "QuantumTech Inc. SEC Form 10-K (FY2024):\n"
            "- Total Revenue: $111.3 Million USD.\n"
            "- EBITDA: $21.1 Million USD.\n"
            "- Cash and Equivalents: $22.4 Million USD.\n"
            "- Total Debt Liabilities: $15.5 Million USD."
        ),
        "source": "SEC-10K-FY2024"
    }
]

# 2. Intellectual Property Database (USPTO Patents & Publications)
PATENT_DB = [
    {
        "company": "QuantumTech Inc.",
        "patent_id": "US-11948271-B2",
        "title": "Silicon-integrated superconducting quantum key distribution transceiver",
        "abstract": (
            "Abstract: This invention describes a micro-chip scale hardware transceiver utilizing "
            "silicon photonics to emit entangled photon pairs at telecom wavelengths (1550nm). "
            "Integrates superconducting nanowire single-photon detectors (SNSPDs) on a single silicon substrate. "
            "Enables cryptographic key distribution secure against quantum computer Shor-algorithm decryption."
        ),
        "filing_date": "2024-03-12",
        "source": "USPTO-PAT-US-11948271"
    },
    {
        "company": "QuantumTech Inc.",
        "patent_id": "US-12053910-B1",
        "title": "Cryogenic thermal isolating packaging for multi-qubit processors",
        "abstract": (
            "Abstract: A packaging system comprising nested vacuum-insulated chambers designed to isolate "
            "a 128-qubit processor core from thermal electromagnetic radiation down to 10 milli-Kelvin. "
            "Uses gold-plated copper shielding and superconducting coaxial trace paths to minimize signal cross-talk."
        ),
        "filing_date": "2025-01-08",
        "source": "USPTO-PAT-US-12053910"
    }
]

# 3. Legal & Regulatory Database (Court Dockets & Compliance Records)
LEGAL_DB = [
    {
        "company": "QuantumTech Inc.",
        "case_id": "FTC-2025-A-928",
        "matter": "Antitrust & Monopoly Review regarding proposed merger with CryoSystems Ltd.",
        "status": "Ongoing / Under Review",
        "details": (
            "Details: FTC Bureau of Competition opened a preliminary investigation in October 2025 "
            "to evaluate whether QuantumTech's proposed $30M acquisition of CryoSystems Ltd. (sole manufacturer "
            "of cryogenic dilution refrigeration valves) constitutes a vertical monopoly in the quantum supply chain. "
            "QuantumTech has filed a response arguing alternative valve fabricators exist in the EU."
        ),
        "source": "FTC-Antitrust-Docket-928"
    },
    {
        "company": "QuantumTech Inc.",
        "case_id": "DEL-CH-10922-2025",
        "matter": "Patent Infringement Lawsuit filed by CyberSec Global Corp.",
        "status": "Active / Pre-trial discovery",
        "details": (
            "Details: CyberSec Global Corp filed a civil suit in Delaware Chancery Court in November 2025, "
            "claiming QuantumTech's silicon-integrated photonics transceiver (Patent US-11948271-B2) infringes "
            "upon CyberSec's core patent US-10822194 covering coherent optical ring resonators. "
            "QuantumTech legal counsel is preparing an invalidity defense, estimating a 75% probability of winning."
        ),
        "source": "Delaware-Chancery-Court-CH-10922"
    }
]


In [ ]:

# ==============================================================================
# Step 2: Define Specialized Retrieval Tools
# ==============================================================================

@tool
def query_sec_financials(company: str, query: str) -> str:
    """Retrieves financial filings, revenues, EBITDA, balance sheets, and debt liabilities for a given company."""
    results = []
    for doc in FINANCIAL_DB:
        if company.lower() in doc["company"].lower() or company.lower() in doc["document"].lower():
            results.append(f"Source: [{doc['source']}]\n{doc['document']}")
    
    if not results:
        return f"No financial documents found matching '{company}'."
    return "\n\n---\n\n".join(results)

@tool
def query_ip_patents(company: str, query: str) -> str:
    """Retrieves patent abstracts, titles, and technical system details for a given company's IP holdings."""
    results = []
    for doc in PATENT_DB:
        if company.lower() in doc["company"].lower() or company.lower() in doc["title"].lower() or company.lower() in doc["abstract"].lower():
            results.append(f"Source: [{doc['source']}]\nPatent ID: {doc['patent_id']}\nTitle: {doc['title']}\n{doc['abstract']}")
            
    if not results:
        return f"No intellectual property found matching '{company}'."
    return "\n\n---\n\n".join(results)

@tool
def query_legal_compliance(company: str, query: str) -> str:
    """Retrieves lawsuits, regulatory compliance audits, FTC investigations, and court dockets for a given company."""
    results = []
    for doc in LEGAL_DB:
        if company.lower() in doc["company"].lower() or company.lower() in doc["matter"].lower() or company.lower() in doc["details"].lower():
            results.append(f"Source: [{doc['source']}]\nCase ID: {doc['case_id']}\nMatter: {doc['matter']}\nStatus: {doc['status']}\n{doc['details']}")
            
    if not results:
        return f"No legal/compliance cases found matching '{company}'."
    return "\n\n---\n\n".join(results)



In [ ]:


# ==============================================================================
# Step 3: Build Specialized Retrieval Worker Agents
# ==============================================================================

# We compile separate ReAct agents for each domain. These are completely decoupled,
# meaning we can scale their tools, prompt structures, or models independently!
financial_agent = create_react_agent(
    worker_model,
    tools=[query_sec_financials],
    prompt=(
        "You are an expert Financial Auditor and SEC RAG specialist. Your task is to query "
        "the financial records for the requested company, extract revenues, margin, EBITDA, "
        "debt, and financial risks, and formulate a highly structured Financial Briefing. "
        "Always cite your sources exactly using [SEC-10K-FYXXXX] format."
    )
)

ip_agent = create_react_agent(
    worker_model,
    tools=[query_ip_patents],
    prompt=(
        "You are an expert Patent Analyst and Tech RAG specialist. Your task is to query "
        "patent databases for the requested company, outline their key hardware and optical claims, "
        "and explain the technical mechanics of their integrated circuits. "
        "Always cite your sources exactly using [USPTO-PAT-US-XXXXXX] format."
    )
)

legal_agent = create_react_agent(
    worker_model,
    tools=[query_legal_compliance],
    prompt=(
        "You are a Corporate Legal Counsel and Compliance RAG specialist. Your task is to query "
        "litigation dockets and FTC filings for the requested company. Summarize active lawsuits, "
        "antitrust issues, ongoing audits, and estimate their legal risks. "
        "Always cite your sources exactly using [Docket-ID] format."
    )
)


In [ ]:

# ==============================================================================
# Step 4: Define Parent Graph State and Structured Supervisor Router
# ==============================================================================

class DiligenceState(MessagesState):
    """The central state for the Orchestrator graph."""
    # Tracks the supervisor's dynamic routing decisions
    next_agent: str
    sub_query: str
    
    # Store the synthesized reports from the individual workers
    # (Information Hiding: the Supervisor reviews these summaries, not the raw doc chunks)
    financial_report: str
    ip_report: str
    legal_report: str


# Pydantic schema for Supervisor's structured routing decisions
class RouterDecision(BaseModel):
    next: Literal["financial_worker", "ip_worker", "legal_worker", "FINISH"] = Field(
        description="Select the next domain specialist to consult. Select FINISH only when you have collected all necessary briefings."
    )
    sub_query: str = Field(
        description="The tailored query/question passed to the specialist worker. Keep it focused on their specific domain."
    )
    reasoning: str = Field(
        description="Explain why you are choosing this next step or why you have sufficient data to FINISH."
    )

# Instantiate structured router
structured_supervisor_router = orchestrator_model.with_structured_output(RouterDecision)



In [ ]:

# ==============================================================================
# Step 5: Implement Graph Nodes (Supervisor & Worker Wrappers)
# ==============================================================================

def deal_supervisor(state: DiligenceState) -> dict:
    """The high-level orchestrator node that decomposes the task, manages routing, and plans next steps."""
    system_prompt = SystemMessage(content=(
        "You are the M&A Due Diligence Deal Supervisor (Orchestrator). You coordinate an audit of QuantumTech Inc. "
        "You manage three specialized retrieval worker agents:\n"
        "- financial_worker: Expert in balance sheets, revenues, cash, and EBITDA.\n"
        "- ip_worker: Expert in patent abstracts and technical hardware claims.\n"
        "- legal_worker: Expert in court lawsuits, FTC investigations, and regulatory actions.\n\n"
        "Your task is to analyze the target company across all three dimensions. "
        "Do not write or guess the facts yourself. Route requests to the workers one by one to gather facts. "
        "Formulate clear, specific queries in the 'sub_query' field.\n"
        "Once you have gathered reports from all necessary domains, select next='FINISH' to write your unified final report."
    ))
    
    # Call structured LLM
    decision = structured_supervisor_router.invoke([system_prompt] + state["messages"])
    
    print(f"\n[Supervisor] Routing to: {decision.next.upper()}")
    print(f"   Reasoning: {decision.reasoning}")
    if decision.next != "FINISH":
        print(f"   Formulated Query: '{decision.sub_query}'")
        
    return {
        "next_agent": decision.next,
        "sub_query": decision.sub_query,
        # Append supervisor's reasoning trace to state messages
        "messages": [AIMessage(content=f"[Supervisor Planning]: Routing to {decision.next}. Target search: {decision.sub_query}", name="DealSupervisor")]
    }

# -- Information Hiding Wrappers --------------------------
# Rather than adding raw retrieval results directly to the supervisor's context,
# these wrappers execute the worker agent, grab the final synthesized summary,
# store it in a dedicated State field, and return only the high-level summary to the supervisor's message log.

def run_financial_worker(state: DiligenceState) -> dict:
    query = state["sub_query"]
    print(f"   [Financial Worker] Consulting Financial Database for: '{query}'...")
    
    # Execute the react agent
    result = financial_agent.invoke({"messages": [HumanMessage(content=query)]})
    final_brief = result["messages"][-1].content
    
    return {
        "messages": [AIMessage(content=f"[FINANCIAL REPORT BRIEFING]:\n{final_brief}", name="FinancialWorker")],
        "financial_report": final_brief
    }

def run_ip_worker(state: DiligenceState) -> dict:
    query = state["sub_query"]
    print(f"   [IP Worker] Consulting Patent & Tech Database for: '{query}'...")
    
    # Execute the react agent
    result = ip_agent.invoke({"messages": [HumanMessage(content=query)]})
    final_brief = result["messages"][-1].content
    
    return {
        "messages": [AIMessage(content=f"[IP & PATENT REPORT BRIEFING]:\n{final_brief}", name="IPWorker")],
        "ip_report": final_brief
    }

def run_legal_worker(state: DiligenceState) -> dict:
    query = state["sub_query"]
    print(f"   [Legal Worker] Consulting Court Dockets for: '{query}'...")
    
    # Execute the react agent
    result = legal_agent.invoke({"messages": [HumanMessage(content=query)]})
    final_brief = result["messages"][-1].content
    
    return {
        "messages": [AIMessage(content=f"[LEGAL & COMPLIANCE BRIEFING]:\n{final_brief}", name="LegalWorker")],
        "legal_report": final_brief
    }

def synthesize_final_report(state: DiligenceState) -> dict:
    """Final node that compiles all retrieved domain briefings into a beautiful corporate due diligence summary."""
    print("\n[Supervisor] Compiling comprehensive investment audit...")
    
    prompt = (
        "You are the Deal Supervisor. You have gathered individual briefings from the Financial, IP, and Legal expert worker agents.\n"
        "Compile a comprehensive, beautifully structured M&A Due Diligence Audit Report for QuantumTech Inc.\n\n"
        f"1. FINANCIAL REPORT REPORT SUMMARY:\n{state['financial_report']}\n\n"
        f"2. INTELLECTUAL PROPERTY PATENT SUMMARY:\n{state['ip_report']}\n\n"
        f"3. LEGAL & COMPLIANCE SUMMARY:\n{state['legal_report']}\n\n"
        "Draft a cohesive report including an executive summary, a breakdown of findings for each department, "
        "an M&A risk assessment grade (e.g. Low, Medium, High Risk), and a final investment recommendation.\n"
        "Crucial Requirement: You must maintain all source citations (e.g., [SEC-10K-FY2025], [USPTO-PAT-US-XXXXXX]) exactly as provided by the workers."
    )
    
    response = orchestrator_model.invoke([SystemMessage(content=prompt)] + state["messages"])
    
    return {"messages": [AIMessage(content=response.content, name="DealSupervisor")]}



In [ ]:

# ==============================================================================
# Step 6: Assemble and Compile the Graph
# ==============================================================================

def route_next(state: DiligenceState) -> str:
    """Conditional routing function."""
    if state["next_agent"] == "financial_worker":
        return "financial_worker"
    elif state["next_agent"] == "ip_worker":
        return "ip_worker"
    elif state["next_agent"] == "legal_worker":
        return "legal_worker"
    return "synthesize"

workflow = StateGraph(DiligenceState)

# Add Nodes
workflow.add_node("supervisor", deal_supervisor)
workflow.add_node("financial_worker", run_financial_worker)
workflow.add_node("ip_worker", run_ip_worker)
workflow.add_node("legal_worker", run_legal_worker)
workflow.add_node("synthesize", synthesize_final_report)

# Set edges
workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_next,
    {
        "financial_worker": "financial_worker",
        "ip_worker": "ip_worker",
        "legal_worker": "legal_worker",
        "synthesize": "synthesize"
    }
)
workflow.add_edge("financial_worker", "supervisor")
workflow.add_edge("ip_worker", "supervisor")
workflow.add_edge("legal_worker", "supervisor")
workflow.add_edge("synthesize", END)

# Compile Graph
compiled_workflow = workflow.compile()

compiled_workflow

In [ ]:

# ==============================================================================
# Step 7: Run the System
# ==============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("HIERARCHICAL MULTI-AGENT RAG (M&A DUE DILIGENCE ENGINE) - Starting")
    print("=" * 80)
    
    # A complex user prompt requiring retrieval across multiple heterogeneous sources
    user_query = (
        "We are evaluating a potential M&A deal with QuantumTech Inc. "
        "Retrieve and audit their recent 2025 financial figures, check if their patent holdings "
        "support their silicon-photonics hardware claims, and audit their litigation dockets "
        "for active lawsuits or FTC antitrust problems. Assemble a cohesive investment review."
    )
    
    print(f"User Inquiry:\n\"{user_query}\"\n")
    
    # Run the workflow
    # We increase the recursion_limit because the graph loops back to the supervisor
    inputs = {"messages": [HumanMessage(content=user_query)]}
    result = compiled_workflow.invoke(inputs, {"recursion_limit": 50})
    
    print("\n" + "=" * 80)
    print("FINAL DUE DILIGENCE REPORT")
    print("=" * 80)
    print(result["messages"][-1].content)
    print("=" * 80)
    
    # Verify the "Information Hiding" benefit (Context window footprint)
    print("\nCONTEXT FOOTPRINT ANALYSIS (Information Hiding Demonstration)")
    print("=" * 80)
    print("This shows that the Supervisor only stored domain briefings, not the raw retrieved vector chunks:")
    

In [ ]:

    total_chars = 0
    for idx, msg in enumerate(result["messages"]):
        name = getattr(msg, "name", None) or "User"
        role = "Orchestrator" if name == "DealSupervisor" else ("Worker" if "Worker" in name else "User")
        content_len = len(msg.content) if msg.content else 0
        total_chars += content_len
        print(f"[{idx+1:02d}] {name:<18} ({role:<12}): Message Length = {content_len:<4} characters")
        
    print(f"\nTotal characters kept in active message list: {total_chars}")
    print("=" * 80 + "\n")